<a href="https://colab.research.google.com/github/elviejoguille/learning/blob/feature%2Fhf-notebooks/HuggingFace_FineTune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers evaluate datasets accelerate -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 59.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.4/81.4 kB 7.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 486.2/486.2 kB 36.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.2/244.2 kB 15.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.8/268.8 kB 17.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 94.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 38.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 7.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.5/212.5 kB 14.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.3/134.3 kB 6.4 MB/s eta 0:00:00


### 1 - Datasets

[Repositorio de Datasets](https://huggingface.co/datasets)

In [ ]:
from datasets import list_datasets, load_dataset, DatasetInfo

all_datasets = list_datasets()
print(f"Existen {len(all_datasets)} datasets")

<ipython-input-1-b700567ac15e>:3: FutureWarning: list_datasets is deprecated and will be removed in the next major version of datasets. Use 'huggingface_hub.list_datasets' instead.
  all_datasets = list_datasets()


Existen 46173 datasets


In [ ]:
all_datasets[:5]

['acronym_identification',
 'ade_corpus_v2',
 'adversarial_qa',
 'aeslc',
 'afrikaans_ner_corpus']

Nueva función para listar los datasets

In [ ]:
from huggingface_hub import list_datasets, dataset_info

all_datasets = list_datasets(sort="downloads", direction=-1, limit=5)

In [ ]:
# next(all_datasets)

In [ ]:
# dataset_info('acronym_identification')

### 2 - Cargar los datos

In [ ]:
dataset = load_dataset("yelp_review_full")
dataset

Generating train split:   0%|          | 0/650000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Dataset yelp_review_full downloaded and prepared to /root/.cache/huggingface/datasets/yelp_review_full/yelp_review_full/1.0.0/e8e18e19d7be9e75642fc66b198abadb116f73599ec89a69ba5dd8d1e57ba0bf. Subsequent calls will reuse this data.


  0%|          | 0/2 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'text'],
        num_rows: 650000
    })
    test: Dataset({
        features: ['label', 'text'],
        num_rows: 50000
    })
})

In [ ]:
dataset["train"][42]

{'label': 4,
 'text': 'What a find! I stopped in here for breakfast while in town for business. The service is so friendly I thought I was down south. The service was quick, frankly and felt like I was with family. \\nFantastic poached eggs, Cajun homefries and crispy bacon. Gab and Eat is definitely a place I world recommend to locals. I was stuffed and the bill was only $8.00.'}

In [ ]:
# para un mejor entrenamiento utilizar más datos
small_train_dataset = dataset["train"].shuffle(seed=42).select(range(80))
small_eval_dataset = dataset["test"].shuffle(seed=42).select(range(40))

### 3 - Tokenizer

In [ ]:
from transformers import AutoTokenizer

In [ ]:
modelo = "bert-base-cased"

tokenizer = AutoTokenizer.from_pretrained(modelo)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

small_train_dataset = small_train_dataset.map(tokenize_function, batched=True)
small_eval_dataset = small_eval_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/80 [00:00<?, ? examples/s]

Map:   0%|          | 0/40 [00:00<?, ? examples/s]

### 4 - Modelo

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(modelo, num_labels=5)

Some weights of the model checkpoint at bert-base-cased were not used when initializing BertForSequenceClassification: ['cls.predictions.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.bias']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initi

### 5 - Entrenamiento

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
import numpy as np
import evaluate

In [ ]:
metric = evaluate.load("accuracy")

In [ ]:
# calculamos el accuracy de las predicciones
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [ ]:
from transformers import TrainingArguments, Trainer

In [ ]:
# definimos los argumentos del entrenamiento
training_args = TrainingArguments(
    'mi-super-modelo',
    evaluation_strategy="steps",
    logging_steps=5,
    num_train_epochs=1,
    push_to_hub=True,
)

[Argumentos Training](https://huggingface.co/transformers/v4.2.2/main_classes/trainer.html)

In [ ]:
# instanciamos el objeto Trainer con todo lo que hemos preparado
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
    compute_metrics=compute_metrics,
)

Cloning https://huggingface.co/NechuBM/mi-super-modelo into local empty directory.


Entrenamos...

In [ ]:
# 8 min con 80 ejemplos (aprox)
trainer.train()

/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Step,Training Loss,Validation Loss,Accuracy
5,1.705800,1.704634,0.225000
10,1.620800,1.640423,0.225000


TrainOutput(global_step=10, training_loss=1.663292407989502, metrics={'train_runtime': 667.9596, 'train_samples_per_second': 0.12, 'train_steps_per_second': 0.015, 'total_flos': 21049451397120.0, 'train_loss': 1.663292407989502, 'epoch': 1.0})

#### Guardar el modelo en local

In [ ]:
#  trainer.save_model("./modelo")

#### Subir a HuggingFace

In [ ]:
trainer.push_to_hub()

Upload file pytorch_model.bin:   0%|          | 1.00/413M [00:00<?, ?B/s]

Upload file training_args.bin:   0%|          | 1.00/3.81k [00:00<?, ?B/s]

Upload file runs/Jul14_10-25-25_9728fd62ed0b/events.out.tfevents.1689330345.9728fd62ed0b.708.0:   0%|         …

To https://huggingface.co/NechuBM/mi-super-modelo
   ffb3534..1d4745a  main -> main

   ffb3534..1d4745a  main -> main

To https://huggingface.co/NechuBM/mi-super-modelo
   1d4745a..2499fa9  main -> main

   1d4745a..2499fa9  main -> main



'https://huggingface.co/NechuBM/mi-super-modelo/commit/1d4745a50ff39ea486fa10e6f02f466a52dac6f1'

In [ ]:
tokenizer.push_to_hub("NechuBM/mi-super-modelo")

CommitInfo(commit_url='https://huggingface.co/NechuBM/mi-super-modelo/commit/34747d1f6131b88a567b5c67b68b37a733467ec7', commit_message='Upload tokenizer', commit_description='', oid='34747d1f6131b88a567b5c67b68b37a733467ec7', pr_url=None, pr_revision=None, pr_num=None)